# Aligned vessel + ETDRS/Disc zones — timeline 시각화

파이프라인 (`scripts/run_pipeline.py` or `run_compare.py`) 이 저장한 산출물에서
**한 환자·한 눈**의 모든 visit 을 가로로 나열해서:
- 배경: preprocessed RGB (dim)
- 초록: aligned vessel mask (fixed frame 기준)
- 반투명 컬러: ETDRS 9 + Zone B/C 오버레이 (모든 visit 동일 — fixed frame 이니까)

**전제**: 파이프라인을 `--keep-masks` 로 돌렸어야 함 (기본은 삭제).
`--drop-masks` 로 이미 지운 환자라면 그 환자만 `--force` + `--keep-masks` 로 재추출.

**필요 파일**:
- `<output_dir>/features_512_aligned/<pid>.json` — meta + features
- `<output_dir>/preprocessed_512/<pid>/rgb/*.png`
- `<output_dir>/aligned_512/<pid>/<eye>/vessels/*.png`

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Patch
from PIL import Image

PKG_ROOT = Path('..').resolve()
if str(PKG_ROOT) not in sys.path:
    sys.path.insert(0, str(PKG_ROOT))

from features.zones import (
    make_etdrs_9subfields, make_disc_zones,
    ETDRS_9_NAMES, DISC_ZONE_NAMES,
    DEFAULT_ETDRS_RADII_MM, DEFAULT_DISC_MARGIN_ZONES,
)

## 1. 대상 지정 — output_dir + 환자 + 눈

`SOURCE = 'notebook'` : 01-alignment + 02-features-align 결과 (src_name 하위)
`SOURCE = 'script'`   : scripts/run_compare.py 결과 (compare_run 하위, src_name 없음)


In [ ]:
# ═════ SOURCE 선택 ═════
SOURCE      = "script"   # "notebook" or "script"

OUTPUT_DIR  = Path("g:/RAG/project_1/output")
SQUARE_SIZE = 512
SRC_NAME    = "20230118_HCS_Fundus_image"   # notebook 모드에서만 사용
COMPARE_RUN = "compare_run"                  # script 모드에서만 사용

PATIENT_ID  = "30221776"    # ← 대상 환자
EYE         = "L"           # "L" or "R"
MAX_TIMELINE = 12           # 너무 많으면 등간격 sampling

# ── 파생 경로 (SOURCE 에 따라 분기) ──
if SOURCE == "notebook":
    features_root = OUTPUT_DIR / f"features_json_aligned_{SQUARE_SIZE}" / SRC_NAME
    prep_root     = OUTPUT_DIR / f"preprocessed_{SQUARE_SIZE}" / SRC_NAME
    aligned_root  = OUTPUT_DIR / f"aligned_{SQUARE_SIZE}" / SRC_NAME
    rgb_dir       = prep_root / "rgb"
    vess_dir      = aligned_root / PATIENT_ID / EYE / "vessels"
elif SOURCE == "script":
    run_root      = OUTPUT_DIR / COMPARE_RUN
    features_root = run_root / f"features_{SQUARE_SIZE}_aligned"
    prep_root     = run_root / f"preprocessed_{SQUARE_SIZE}"
    aligned_root  = run_root / f"aligned_{SQUARE_SIZE}"
    rgb_dir       = prep_root / PATIENT_ID / "rgb"
    vess_dir      = aligned_root / PATIENT_ID / EYE / "vessels"
else:
    raise ValueError(f"SOURCE 는 'notebook' 또는 'script' - got {SOURCE!r}")

json_path = features_root / f"{PATIENT_ID}.json"

for pth, label in [(json_path, "features JSON"),
                    (rgb_dir,  "preprocessed RGB dir"),
                    (vess_dir, "aligned vessel mask dir")]:
    ok = pth.exists()
    print(f'  {"[OK]" if ok else "[!!]"} {label}: {pth}')
    assert ok, f"{label} 없음 - SOURCE={SOURCE!r} 로 파이프라인 돌렸나 확인"


## 2. JSON 로드 → visit 리스트 + zone 재구성

In [ ]:
with open(json_path, encoding='utf-8') as f:
    payload = json.load(f)

feats = pd.DataFrame(payload['features'])
feats['date'] = pd.to_datetime(feats['date'])
eye_feats = feats[feats['eye'] == EYE].sort_values('date').reset_index(drop=True)
print(f'{PATIENT_ID}/{EYE}: 총 {len(eye_feats)} visits (date {eye_feats["date"].min().date()} → {eye_feats["date"].max().date()})')

meta = payload['meta_by_eye'][EYE]
fx, fy   = meta['fovea_x'], meta['fovea_y']
dcx, dcy = meta['disc_cx'], meta['disc_cy']
dd       = meta['disc_diameter_px']
mpp      = meta['mm_per_px']
print(f'  fovea=({fx:.1f}, {fy:.1f})  disc=({dcx:.1f}, {dcy:.1f})  DD={dd:.1f}px  mm/px={mpp:.4f}')

# ── zones (fixed frame — 모든 visit 동일) ──
etdrs  = make_etdrs_9subfields(fx, fy, EYE, mpp, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_ETDRS_RADII_MM)
disc_z = make_disc_zones(dcx, dcy, dd, (SQUARE_SIZE, SQUARE_SIZE), DEFAULT_DISC_MARGIN_ZONES)
zones  = {**etdrs, **disc_z}
print(f'  zones: {list(zones.keys())}')

## 3. Overlay helper — RGB dim + vessel green + zones semi-transparent

In [ ]:
ZONE_COLORS = {
    # ETDRS 9-subfield (fovea-centered)
    'C':  [255, 255, 100],   'S1': [130, 200, 255], 'N1': [180, 130, 255],
    'I1': [130, 255, 180],   'T1': [255, 180, 130], 'S2': [80,  130, 200],
    'N2': [130, 80,  200],   'I2': [80,  200, 130], 'T2': [200, 130, 80],
    # Disc-centered (Wong)
    'zone_B': [255, 100, 100],
    'zone_C': [180, 60,  60],
}

def make_composite(rgb, vessel_mask, zones,
                    rgb_dim=0.35, zone_alpha=0.30, vessel_color=(20, 255, 20)):
    '''Timeline 한 프레임: RGB dim + zones overlay + vessel green.'''
    disp = (rgb.astype(np.float32) * rgb_dim).clip(0, 255)
    # zones
    for name, mask in zones.items():
        c = np.array(ZONE_COLORS.get(name, [200, 200, 200]), dtype=np.float32)
        disp[mask] = disp[mask] * (1 - zone_alpha) + c * zone_alpha
    # vessel (aligned)
    vm = vessel_mask > 0
    if vm.any():
        disp[vm] = np.array(vessel_color, dtype=np.float32)
    return disp.clip(0, 255).astype(np.uint8)

def sample_indices(n, k):
    '''n 개 visit 에서 등간격 k 개 인덱스 (첫/끝 포함).'''
    if n <= k:
        return list(range(n))
    idx = np.linspace(0, n - 1, k).round().astype(int)
    return sorted(set(idx.tolist()))

## 4. Timeline 그리기

너무 많으면 `MAX_TIMELINE` 개 등간격 sampling.

In [ ]:
idxs = sample_indices(len(eye_feats), MAX_TIMELINE)
n_show = len(idxs)
print(f'전체 {len(eye_feats)} visits 중 {n_show} 개 표시')

fig, axes = plt.subplots(1, n_show, figsize=(2.5 * n_show, 2.7), squeeze=False)
for ax, i in zip(axes.flat, idxs):
    row  = eye_feats.iloc[i]
    vid  = row['id']
    date = row['date'].date()

    rgb_p  = rgb_dir  / f'{vid}.png'
    vess_p = vess_dir / f'{vid}.png'
    if not (rgb_p.exists() and vess_p.exists()):
        ax.set_title(f'{date}\n(missing)', fontsize=7); ax.axis('off'); continue

    rgb = np.array(Image.open(rgb_p))
    vm  = np.array(Image.open(vess_p))
    ax.imshow(make_composite(rgb, vm, zones))
    ax.set_title(f'{date}\nvisit {i+1}/{len(eye_feats)}', fontsize=7)
    ax.axis('off')

handles = [Patch(color=np.array(c)/255, label=n) for n, c in ZONE_COLORS.items()]
handles.append(Patch(color='lime', label='aligned vessel'))
fig.legend(handles=handles, loc='lower center', ncol=len(handles),
            bbox_to_anchor=(0.5, -0.05), fontsize=8, frameon=False)
fig.suptitle(f'Patient {PATIENT_ID}/{EYE} — aligned vessel + ETDRS/Disc zones '
              f'(zone 고정 · {n_show} of {len(eye_feats)} visits)', fontsize=11)
plt.tight_layout(rect=(0, 0.02, 1, 0.98)); plt.show()

## 5. (옵션) 하나 크게 보기 — 특정 visit index

In [ ]:
PICK = 0    # 0 = fixed (첫 visit)

row = eye_feats.iloc[PICK]
vid, date = row['id'], row['date'].date()
rgb = np.array(Image.open(rgb_dir  / f'{vid}.png'))
vm  = np.array(Image.open(vess_dir / f'{vid}.png'))

fig, ax = plt.subplots(figsize=(7, 7))
ax.imshow(make_composite(rgb, vm, zones))
ax.scatter(fx, fy, c='white', marker='x', s=120, linewidths=2, label='fovea')
ax.scatter(dcx, dcy, c='white', marker='o', s=120, facecolors='none', linewidths=2, label='disc')
ax.set_title(f'{PATIENT_ID}/{EYE}  visit {PICK+1}/{len(eye_feats)}  ({date})')
ax.axis('off'); ax.legend(loc='lower right', fontsize=9)

handles = [Patch(color=np.array(c)/255, label=n) for n, c in ZONE_COLORS.items()]
handles.append(Patch(color='lime', label='aligned vessel'))
fig.legend(handles=handles, loc='lower center', ncol=6, bbox_to_anchor=(0.5, -0.02), fontsize=9)
plt.tight_layout(); plt.show()